# 07 Hong Kong Observatory Settlement Source Retrieval

## Purpose

This notebook checks whether Hong Kong Observatory data can be used as the official realised-temperature source for Hong Kong temperature markets.

The purpose is to replace proxy realised data with a settlement-source-consistent value. The relevant Polymarket Hong Kong temperature rules refer to the Hong Kong Observatory and the “Absolute Daily Max (deg. C)” in the HKO Daily Extract. The Hong Kong open data portal also provides daily maximum temperature data for the Hong Kong Observatory station through a CSV/API route.

The output of this notebook is a settlement table containing the contract date, official source, realised daily maximum temperature, source route and retrieval status. This table will be used later for time-aligned probability scoring, supervised threshold classification and trading evaluation.

## 1. Imports and paths

In [3]:
from pathlib import Path
from datetime import datetime, timezone
from io import StringIO
import re

import requests
import pandas as pd

RAW_DIR = Path("../data/raw/hko")
PROCESSED_DIR = Path("../data/processed/hko")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 80)

print("Notebook run time UTC:", datetime.now(timezone.utc).isoformat())
print("Raw directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)

Notebook run time UTC: 2026-06-16T23:21:56.970418+00:00
Raw directory: ../data/raw/hko
Processed directory: ../data/processed/hko


## 2. Contract metadata

The first test case uses the Hong Kong temperature market previously used in the single-market prototype. The date and slug can be changed later when a broader event table is constructed.

In [5]:
contract_metadata = {
    "city": "Hong Kong",
    "contract_slug": "highest-temperature-in-hong-kong-on-may-30-2026",
    "polymarket_event_url": "https://polymarket.com/event/highest-temperature-in-hong-kong-on-may-30-2026",
    "settlement_date_local": "2026-05-30",
    "settlement_source": "Hong Kong Observatory",
    "settlement_variable_expected": "Absolute Daily Max (deg. C)",
    "station": "HKO",
    "unit": "deg C",
}

contract_metadata

{'city': 'Hong Kong',
 'contract_slug': 'highest-temperature-in-hong-kong-on-may-30-2026',
 'polymarket_event_url': 'https://polymarket.com/event/highest-temperature-in-hong-kong-on-may-30-2026',
 'settlement_date_local': '2026-05-30',
 'settlement_source': 'Hong Kong Observatory',
 'settlement_variable_expected': 'Absolute Daily Max (deg. C)',
 'station': 'HKO',
 'unit': 'deg C'}

## 3. HKO open data API: daily maximum temperature

The Hong Kong open data route provides daily maximum temperature data for the Hong Kong Observatory station.

The all-year route is checked first because it is the most stable route for historical values. The current-year route is also checked because it may contain recent values once the relevant month is updated.

In [7]:
def fetch_text(url, timeout=30):
    try:
        response = requests.get(url, timeout=timeout)
        return {
            "url": url,
            "status_code": response.status_code,
            "ok": response.ok,
            "content_type": response.headers.get("content-type"),
            "text": response.text,
            "error": None,
        }
    except Exception as e:
        return {
            "url": url,
            "status_code": None,
            "ok": False,
            "content_type": None,
            "text": "",
            "error": repr(e),
        }

hko_open_data_urls = {
    "all_year_daily_max": "https://data.weather.gov.hk/weatherAPI/opendata/opendata.php?dataType=CLMMAXT&rformat=csv&station=HKO",
    "year_2026_daily_max": "https://data.weather.gov.hk/weatherAPI/opendata/opendata.php?dataType=CLMMAXT&year=2026&rformat=csv&station=HKO",
}

open_data_results = {}

for name, url in hko_open_data_urls.items():
    result = fetch_text(url)
    open_data_results[name] = result
    print(name, result["status_code"], result["content_type"], "chars:", len(result["text"]), "error:", result["error"])
    (RAW_DIR / f"{name}.csv").write_text(result["text"], encoding="utf-8")

all_year_daily_max 200 text/csv; charset=utf-8 chars: 838759 error: None
year_2026_daily_max 200 text/csv; charset=utf-8 chars: 2733 error: None


## 4. Parse HKO CSV data

The HKO CSV may contain metadata rows or column names that differ across routes. The parser therefore tries several header positions and then standardises the date and daily maximum temperature fields.

In [9]:
def try_parse_csv(text, max_skiprows=12):
    parsed_attempts = []
    for skiprows in range(max_skiprows + 1):
        try:
            df = pd.read_csv(StringIO(text), skiprows=skiprows)
            parsed_attempts.append((skiprows, df))
            if df.shape[0] > 0 and df.shape[1] >= 2:
                return df, skiprows, parsed_attempts
        except Exception:
            continue
    raise ValueError("No usable CSV parse found.")

parsed_open_data = {}

for name, result in open_data_results.items():
    if result["ok"] and result["text"].strip():
        try:
            df, skiprows, attempts = try_parse_csv(result["text"])
            parsed_open_data[name] = {"df": df, "skiprows": skiprows}
            print(f"{name}: parsed with skiprows={skiprows}, shape={df.shape}")
            display(df.head())
            display(df.tail())
        except Exception as e:
            print(f"{name}: parse failed:", repr(e))
    else:
        print(f"{name}: request failed or empty response")

all_year_daily_max: parsed with skiprows=2, shape=(49463, 5)


,年/Year,月/Month,日/Day,數值/Value,數據完整性/data Completeness
0,1884,1.0,1.0,15.3,C
1,1884,1.0,2.0,17.1,C
2,1884,1.0,3.0,19.6,C
3,1884,1.0,4.0,23.2,C
4,1884,1.0,5.0,19.4,C


,年/Year,月/Month,日/Day,數值/Value,數據完整性/data Completeness
49458,2026,5.0,30.0,32.6,C
49459,2026,5.0,31.0,30.9,C
49460,*** 沒有數據/unavailable,NaN,NaN,NaN,NaN
49461,# 數據不完整/data incomplete,NaN,NaN,NaN,NaN
49462,C 數據完整/data Complete,NaN,NaN,NaN,NaN


year_2026_daily_max: parsed with skiprows=2, shape=(154, 5)


,年/Year,月/Month,日/Day,數值/Value,數據完整性/data Completeness
0,2026,1.0,1.0,21.7,C
1,2026,1.0,2.0,17.7,C
2,2026,1.0,3.0,16.5,C
3,2026,1.0,4.0,20.3,C
4,2026,1.0,5.0,22.1,C


,年/Year,月/Month,日/Day,數值/Value,數據完整性/data Completeness
149,2026,5.0,30.0,32.6,C
150,2026,5.0,31.0,30.9,C
151,*** 沒有數據/unavailable,NaN,NaN,NaN,NaN
152,# 數據不完整/data incomplete,NaN,NaN,NaN,NaN
153,C 數據完整/data Complete,NaN,NaN,NaN,NaN


In [10]:
def normalise_col(col):
    return re.sub(r"[^a-z0-9]+", "_", str(col).strip().lower()).strip("_")

def standardise_hko_daily_max(df, source_name):
    out = df.copy()
    out.columns = [normalise_col(c) for c in out.columns]

    required_ymd = {"year", "month", "day"}

    if required_ymd.issubset(set(out.columns)):
        date_series = pd.to_datetime(
            {
                "year": pd.to_numeric(out["year"], errors="coerce"),
                "month": pd.to_numeric(out["month"], errors="coerce"),
                "day": pd.to_numeric(out["day"], errors="coerce"),
            },
            errors="coerce",
        )
        raw_date_column = "year_month_day"
    else:
        date_candidates = [c for c in out.columns if c in ["date", "data", "yyyymmdd"] or "date" in c]
        if date_candidates:
            date_col = date_candidates[0]
        else:
            date_col = out.columns[0]
        date_series = pd.to_datetime(out[date_col], errors="coerce")
        raw_date_column = date_col

    temp_candidates = [
        c for c in out.columns
        if ("max" in c and ("temp" in c or "temperature" in c))
        or ("maximum" in c and ("temp" in c or "temperature" in c))
        or c in ["value", "temperature", "temp"]
    ]

    if temp_candidates:
        temp_col = temp_candidates[0]
    else:
        numeric_scores = {}
        for c in out.columns:
            if c in ["year", "month", "day"]:
                continue
            numeric_scores[c] = pd.to_numeric(out[c], errors="coerce").notna().sum()
        temp_col = max(numeric_scores, key=numeric_scores.get)

    standardised = pd.DataFrame({
        "date": date_series,
        "official_daily_max_temp_c": pd.to_numeric(out[temp_col], errors="coerce"),
    })

    standardised = standardised.dropna(subset=["date", "official_daily_max_temp_c"])
    standardised["date"] = standardised["date"].dt.date.astype(str)
    standardised["station"] = "HKO"
    standardised["source_name"] = source_name
    standardised["source_route"] = "HKO open data CSV"
    standardised["raw_date_column"] = raw_date_column
    standardised["raw_temperature_column"] = temp_col

    return standardised[
        [
            "date",
            "official_daily_max_temp_c",
            "station",
            "source_name",
            "source_route",
            "raw_date_column",
            "raw_temperature_column",
        ]
    ]


standardised_open_data = {}

for name, obj in parsed_open_data.items():
    try:
        sdf = standardise_hko_daily_max(obj["df"], name)
        standardised_open_data[name] = sdf
        print(name, sdf.shape)
        print("Date range:", sdf["date"].min(), "to", sdf["date"].max())
        display(sdf.head())
        display(sdf.tail())
    except Exception as e:
        print(f"{name}: standardisation failed:", repr(e))

all_year_daily_max (49459, 7)
Date range: 1884-01-01 to 2026-05-31


,date,official_daily_max_temp_c,station,source_name,source_route,raw_date_column,raw_temperature_column
0,1884-01-01,15.3,HKO,all_year_daily_max,HKO open data CSV,year_month_day,value
1,1884-01-02,17.1,HKO,all_year_daily_max,HKO open data CSV,year_month_day,value
2,1884-01-03,19.6,HKO,all_year_daily_max,HKO open data CSV,year_month_day,value
3,1884-01-04,23.2,HKO,all_year_daily_max,HKO open data CSV,year_month_day,value
4,1884-01-05,19.4,HKO,all_year_daily_max,HKO open data CSV,year_month_day,value


,date,official_daily_max_temp_c,station,source_name,source_route,raw_date_column,raw_temperature_column
49455,2026-05-27,33.7,HKO,all_year_daily_max,HKO open data CSV,year_month_day,value
49456,2026-05-28,33.4,HKO,all_year_daily_max,HKO open data CSV,year_month_day,value
49457,2026-05-29,34.1,HKO,all_year_daily_max,HKO open data CSV,year_month_day,value
49458,2026-05-30,32.6,HKO,all_year_daily_max,HKO open data CSV,year_month_day,value
49459,2026-05-31,30.9,HKO,all_year_daily_max,HKO open data CSV,year_month_day,value


year_2026_daily_max (151, 7)
Date range: 2026-01-01 to 2026-05-31


,date,official_daily_max_temp_c,station,source_name,source_route,raw_date_column,raw_temperature_column
0,2026-01-01,21.7,HKO,year_2026_daily_max,HKO open data CSV,year_month_day,value
1,2026-01-02,17.7,HKO,year_2026_daily_max,HKO open data CSV,year_month_day,value
2,2026-01-03,16.5,HKO,year_2026_daily_max,HKO open data CSV,year_month_day,value
3,2026-01-04,20.3,HKO,year_2026_daily_max,HKO open data CSV,year_month_day,value
4,2026-01-05,22.1,HKO,year_2026_daily_max,HKO open data CSV,year_month_day,value


,date,official_daily_max_temp_c,station,source_name,source_route,raw_date_column,raw_temperature_column
146,2026-05-27,33.7,HKO,year_2026_daily_max,HKO open data CSV,year_month_day,value
147,2026-05-28,33.4,HKO,year_2026_daily_max,HKO open data CSV,year_month_day,value
148,2026-05-29,34.1,HKO,year_2026_daily_max,HKO open data CSV,year_month_day,value
149,2026-05-30,32.6,HKO,year_2026_daily_max,HKO open data CSV,year_month_day,value
150,2026-05-31,30.9,HKO,year_2026_daily_max,HKO open data CSV,year_month_day,value


In [11]:
target_date = contract_metadata["settlement_date_local"]

open_data_matches = []

for name, sdf in standardised_open_data.items():
    match = sdf[sdf["date"] == target_date].copy()
    if not match.empty:
        match["source_url"] = hko_open_data_urls[name]
        open_data_matches.append(match)

if open_data_matches:
    open_data_match_df = pd.concat(open_data_matches, ignore_index=True)
else:
    open_data_match_df = pd.DataFrame()

open_data_match_df

,date,official_daily_max_temp_c,station,source_name,source_route,raw_date_column,raw_temperature_column,source_url
0,2026-05-30,32.6,HKO,all_year_daily_max,HKO open data CSV,year_month_day,value,https://data.weather.gov.hk/weatherAPI/opendat...
1,2026-05-30,32.6,HKO,year_2026_daily_max,HKO open data CSV,year_month_day,value,https://data.weather.gov.hk/weatherAPI/opendat...


## 5. HKO Daily Extract page

The HKO Daily Extract page is checked because Polymarket rules may refer directly to the Daily Extract and the “Absolute Daily Max (deg. C)” field.

Automatic parsing may fail if the page structure changes. In that case, the raw HTML is still saved as an audit trail, and the open data CSV provides a reproducible official HKO route.

In [13]:
settlement_date = pd.to_datetime(contract_metadata["settlement_date_local"])
year = settlement_date.year
month = settlement_date.month
target_day = settlement_date.day

daily_extract_urls = {
    "daily_extract_y_m": f"https://www.hko.gov.hk/en/cis/dailyExtract.htm?y={year}&m={month:02d}",
    "daily_extract_m_y": f"https://www.hko.gov.hk/en/cis/dailyExtract.htm?m={month:02d}&y={year}",
}

daily_extract_results = {}

for name, url in daily_extract_urls.items():
    result = fetch_text(url)
    daily_extract_results[name] = result
    print(name, result["status_code"], result["content_type"], "chars:", len(result["text"]), "error:", result["error"])
    (RAW_DIR / f"{name}_{year}_{month:02d}.html").write_text(result["text"], encoding="utf-8")

daily_extract_y_m 200 text/html; charset=utf-8 chars: 20754 error: None
daily_extract_m_y 200 text/html; charset=utf-8 chars: 20754 error: None


In [14]:
daily_extract_tables = []

for name, result in daily_extract_results.items():
    if result["ok"] and result["text"].strip():
        try:
            tables = pd.read_html(StringIO(result["text"]))
            print(name, "tables:", len(tables))
            for i, table in enumerate(tables[:8]):
                print("table", i, "shape", table.shape)
                display(table.head())
                daily_extract_tables.append({
                    "source_name": name,
                    "source_url": result["url"],
                    "table_index": i,
                    "table": table,
                })
        except Exception as e:
            print(name, "read_html failed:", repr(e))

daily_extract_y_m read_html failed: ValueError('No tables found')
daily_extract_m_y read_html failed: ValueError('No tables found')


## 6. Search Daily Extract tables for the target day and maximum temperature column

In [16]:
def flatten_columns(df):
    out = df.copy()
    if isinstance(out.columns, pd.MultiIndex):
        out.columns = [
            " ".join([str(x) for x in col if str(x) != "nan"]).strip()
            for col in out.columns
        ]
    else:
        out.columns = [str(c) for c in out.columns]
    return out

def find_daily_extract_value(table, target_day):
    df = flatten_columns(table)
    df.columns = [normalise_col(c) for c in df.columns]

    # Most Daily Extract tables have day/date in the first column.
    day_col = df.columns[0]

    max_candidates = [
        c for c in df.columns
        if ("absolute" in c and "max" in c)
        or ("daily" in c and "max" in c)
        or ("max" in c and ("temp" in c or "temperature" in c))
        or ("maximum" in c and ("temp" in c or "temperature" in c))
    ]

    if not max_candidates:
        return pd.DataFrame()

    temp_col = max_candidates[0]

    tmp = df.copy()
    tmp["_day_numeric"] = pd.to_numeric(tmp[day_col], errors="coerce")
    tmp = tmp[tmp["_day_numeric"] == target_day].copy()

    if tmp.empty:
        return pd.DataFrame()

    out = tmp[[day_col, temp_col]].copy()
    out.columns = ["day_raw", "absolute_daily_max_temp_c"]
    out["absolute_daily_max_temp_c"] = pd.to_numeric(out["absolute_daily_max_temp_c"], errors="coerce")
    out["date"] = target_date
    out["raw_day_column"] = day_col
    out["raw_temperature_column"] = temp_col

    return out

daily_extract_matches = []

for item in daily_extract_tables:
    match = find_daily_extract_value(item["table"], target_day)
    if not match.empty:
        match["source_name"] = item["source_name"]
        match["source_url"] = item["source_url"]
        match["table_index"] = item["table_index"]
        match["source_route"] = "HKO Daily Extract"
        daily_extract_matches.append(match)

if daily_extract_matches:
    daily_extract_match_df = pd.concat(daily_extract_matches, ignore_index=True)
else:
    daily_extract_match_df = pd.DataFrame()

daily_extract_match_df

""


## 7. Build settlement table

Priority is given to the Daily Extract if the relevant field is parsed automatically. If the Daily Extract is not parsed, the official open data CSV match is used.

The output records the source route explicitly so that later notebooks can distinguish direct Daily Extract retrieval from open data retrieval.

In [18]:
settlement_rows = []

if not daily_extract_match_df.empty and daily_extract_match_df["absolute_daily_max_temp_c"].notna().any():
    for _, row in daily_extract_match_df.dropna(subset=["absolute_daily_max_temp_c"]).iterrows():
        settlement_rows.append({
            "contract_slug": contract_metadata["contract_slug"],
            "polymarket_event_url": contract_metadata["polymarket_event_url"],
            "city": contract_metadata["city"],
            "settlement_date_local": contract_metadata["settlement_date_local"],
            "settlement_source": contract_metadata["settlement_source"],
            "settlement_variable": contract_metadata["settlement_variable_expected"],
            "official_realised_temperature_c": row["absolute_daily_max_temp_c"],
            "station": contract_metadata["station"],
            "unit": contract_metadata["unit"],
            "source_route": row["source_route"],
            "source_reference": row["source_url"],
            "retrieval_status": "matched_daily_extract",
            "retrieval_time_utc": datetime.now(timezone.utc).isoformat(),
        })

elif not open_data_match_df.empty and open_data_match_df["official_daily_max_temp_c"].notna().any():
    for _, row in open_data_match_df.dropna(subset=["official_daily_max_temp_c"]).iterrows():
        settlement_rows.append({
            "contract_slug": contract_metadata["contract_slug"],
            "polymarket_event_url": contract_metadata["polymarket_event_url"],
            "city": contract_metadata["city"],
            "settlement_date_local": contract_metadata["settlement_date_local"],
            "settlement_source": contract_metadata["settlement_source"],
            "settlement_variable": "Daily Maximum Temperature",
            "official_realised_temperature_c": row["official_daily_max_temp_c"],
            "station": contract_metadata["station"],
            "unit": contract_metadata["unit"],
            "source_route": row["source_route"],
            "source_reference": row["source_url"],
            "retrieval_status": "matched_open_data_csv",
            "retrieval_time_utc": datetime.now(timezone.utc).isoformat(),
        })

else:
    settlement_rows.append({
        "contract_slug": contract_metadata["contract_slug"],
        "polymarket_event_url": contract_metadata["polymarket_event_url"],
        "city": contract_metadata["city"],
        "settlement_date_local": contract_metadata["settlement_date_local"],
        "settlement_source": contract_metadata["settlement_source"],
        "settlement_variable": contract_metadata["settlement_variable_expected"],
        "official_realised_temperature_c": None,
        "station": contract_metadata["station"],
        "unit": contract_metadata["unit"],
        "source_route": "not_matched",
        "source_reference": "",
        "retrieval_status": "not_found",
        "retrieval_time_utc": datetime.now(timezone.utc).isoformat(),
    })

settlement_df = pd.DataFrame(settlement_rows)

# If the same official value is found through multiple HKO open data routes,
# keep one settlement observation and preserve all source references.
if not settlement_df.empty and settlement_df["official_realised_temperature_c"].notna().any():
    group_cols = [
        "contract_slug",
        "polymarket_event_url",
        "city",
        "settlement_date_local",
        "settlement_source",
        "settlement_variable",
        "official_realised_temperature_c",
        "station",
        "unit",
        "source_route",
        "retrieval_status",
    ]

    settlement_df = (
        settlement_df
        .groupby(group_cols, dropna=False, as_index=False)
        .agg({
            "source_reference": lambda x: " | ".join(sorted(set(str(v) for v in x if pd.notna(v)))),
            "retrieval_time_utc": "first",
        })
    )

settlement_df

,contract_slug,polymarket_event_url,city,settlement_date_local,settlement_source,settlement_variable,official_realised_temperature_c,station,unit,source_route,retrieval_status,source_reference,retrieval_time_utc
0,highest-temperature-in-hong-kong-on-may-30-2026,https://polymarket.com/event/highest-temperatu...,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,HKO,deg C,HKO open data CSV,matched_open_data_csv,https://data.weather.gov.hk/weatherAPI/opendat...,2026-06-16T23:22:26.556353+00:00


In [19]:
output_path = PROCESSED_DIR / "hko_official_settlement_temperature.csv"
settlement_df.to_csv(output_path, index=False)

print("Saved settlement table:", output_path)
display(settlement_df)

Saved settlement table: ../data/processed/hko/hko_official_settlement_temperature.csv


,contract_slug,polymarket_event_url,city,settlement_date_local,settlement_source,settlement_variable,official_realised_temperature_c,station,unit,source_route,retrieval_status,source_reference,retrieval_time_utc
0,highest-temperature-in-hong-kong-on-may-30-2026,https://polymarket.com/event/highest-temperatu...,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,HKO,deg C,HKO open data CSV,matched_open_data_csv,https://data.weather.gov.hk/weatherAPI/opendat...,2026-06-16T23:22:26.556353+00:00


## 8. Interpretation

The HKO open data route successfully retrieves official Hong Kong Observatory daily maximum temperature data for the selected Hong Kong contract date.

For the current test case, the official realised daily maximum temperature on 2026-05-30 is 32.6°C. This value is retrieved from the Hong Kong Observatory open data CSV route and should replace proxy realised weather data in later scoring and backtesting.

The Daily Extract page is also checked because Polymarket rules refer to the “Absolute Daily Max (deg. C)” in the relevant Daily Extract. Automatic Daily Extract parsing remains unresolved because the page does not expose a simple HTML table through `pandas.read_html`, but the raw HTML is saved for audit.

The source route should be preserved because the final dissertation should distinguish between direct Daily Extract retrieval and the HKO open data CSV route.

The realised value should only be used after the event for scoring. It should not be used when selecting the market price timestamp. The market price must be sampled at the forecast issue time, or at the first available market timestamp after the forecast becomes publicly usable.